In [ ]:
%pip uninstall -y torchao
%pip install -q datasets accelerate sentencepiece evaluate sacrebleu
!pip install -q "transformers==4.46.3" "peft>=0.10.0" "bitsandbytes>=0.43.0"

In [ ]:
import os
import random
import numpy as np
import torch

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

print("PyTorch version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))
    print("Compute Capability:", torch.cuda.get_device_capability(0))
    print("bfloat16 Support:", torch.cuda.is_bf16_supported())
else:
    print("No GPU detected")

In [ ]:
MODEL_TYPE = "mbart"  # Change to "mbart" to train mBART

# Choose model architecture base:
# Options for mT5: "google/mt5-small", "google/mt5-base", "google/mt5-large"
# Options for mBART: "facebook/mbart-large-50-many-to-many-mmt", "facebook/mbart-large-50", "vinai/bartpho-syllable"
MODEL_NAME = "google/mt5" if MODEL_TYPE == "mt5" else "facebook/mbart-large-50-many-to-many-mmt"

# Task & Language settings
TASK_PREFIX = "transliterate Sino-Nom to Vietnamese: " if MODEL_TYPE == "mt5" else ""
SRC_LANG = "zh_CN"  # Used for mBART tokenizer
TGT_LANG = "vi_VN"  # Used for mBART tokenizer

# Data Sequence Lengths
MAX_SOURCE_LEN = 64
MAX_TARGET_LEN = 64

# Training Hyperparameters
BATCH_SIZE = 8
GRAD_ACCUM_STEPS = 2
LEARNING_RATE = 5e-5
NUM_EPOCHS = 5
WARMUP_RATIO = 0.05
WEIGHT_DECAY = 0.01

print(f"Selected Architecture: {MODEL_TYPE.upper()}")
print(f"Selected Model: {MODEL_NAME}")


In [ ]:
from pathlib import Path
import pandas as pd


DATA_DIR = Path("/kaggle/input/datasets/ganth1811/dataset-nom")
OUTPUT_DIR = Path("/kaggle/working/output") if Path("/kaggle/working").exists() else Path("./models/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Data Directory Found: {DATA_DIR.resolve()}")
print(f"Output Directory Set: {OUTPUT_DIR.resolve()}")

train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df = pd.read_csv(DATA_DIR / "val.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

INPUT_COL = "nom"
TARGET_COL = "vietnamese_clean"

train_df = train_df[[INPUT_COL, TARGET_COL]].dropna().reset_index(drop=True)
val_df = val_df[[INPUT_COL, TARGET_COL]].dropna().reset_index(drop=True)
test_df = test_df[[INPUT_COL, TARGET_COL]].dropna().reset_index(drop=True)

print(f"Train samples: {len(train_df):,}")
print(f"Val samples:   {len(val_df):,}")
print(f"Test samples:  {len(test_df):,}")
display(train_df.head(5))

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, BitsAndBytesConfig
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Configure mBART specific language tokens
forced_bos_token_id = None
if MODEL_TYPE == "mbart":
    if hasattr(tokenizer, "src_lang") and hasattr(tokenizer, "tgt_lang"):
        tokenizer.src_lang = SRC_LANG
        tokenizer.tgt_lang = TGT_LANG
    if hasattr(tokenizer, "lang_code_to_id") and TGT_LANG in tokenizer.lang_code_to_id:
        forced_bos_token_id = tokenizer.lang_code_to_id[TGT_LANG]
        print(f"mBART Target Language Set to {TGT_LANG} (forced_bos_token_id: {forced_bos_token_id})")

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
if torch.cuda.is_available():
    model = model.to("cuda:0")

# Set forced BOS token id if mBART
if forced_bos_token_id is not None:
    model.config.forced_bos_token_id = forced_bos_token_id

def print_trainable_parameters(m):
    trainable, total = 0, 0
    for _, p in m.named_parameters():
        total += p.numel()
        if p.requires_grad:
            trainable += p.numel()
    ratio = 100.0 * trainable / total if total > 0 else 0
    print(f"Trainable parameters: {trainable:,} / {total:,} ({ratio:.2f}%)")

print_trainable_parameters(model)

In [ ]:
from datasets import Dataset

train_ds = Dataset.from_pandas(train_df)
val_ds = Dataset.from_pandas(val_df)
test_ds = Dataset.from_pandas(test_df)

def preprocess_function(examples):
    inputs = [TASK_PREFIX + str(x) for x in examples[INPUT_COL]]
    targets = [str(x) for x in examples[TARGET_COL]]
    
    model_inputs = tokenizer(
        inputs,
        max_length=MAX_SOURCE_LEN,
        truncation=True
    )
    
    labels = tokenizer(
        text_target=targets,
        max_length=MAX_TARGET_LEN,
        truncation=True
    )
    
    # Replace pad token ID with -100 to ignore padding in loss calculation
    labels_ids = labels["input_ids"]
    labels_ids = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in labels_ids
    ]
    
    model_inputs["labels"] = labels_ids
    return model_inputs

print("Tokenizing dataset splits...")
tokenized_train = train_ds.map(preprocess_function, batched=True, remove_columns=train_ds.column_names)
tokenized_val = val_ds.map(preprocess_function, batched=True, remove_columns=val_ds.column_names)
print("Dataset tokenization completed successfully!")

In [ ]:
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments, EarlyStoppingCallback
from transformers import PrinterCallback

                
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8
)

training_args = Seq2SeqTrainingArguments(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    logging_steps=20,
    logging_strategy="steps",
    logging_first_step=True,     
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LEN,
    fp16=not torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False,
    bf16=torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
    data_collator=data_collator,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=3)
    ]
)

trainer.add_callback(PrinterCallback())
print("Starting Model Training...")
train_result = trainer.train()
print("Training Finished!")

# Save Final Weights / Adapter
FINAL_MODEL_DIR = OUTPUT_DIR / "final_model"
FINAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)
trainer.model.save_pretrained(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)
print(f"Saved fine-tuned model / adapter to: {FINAL_MODEL_DIR.resolve()}")

In [ ]:
import evaluate
from tqdm import tqdm

model = AutoModelForSeq2SeqLM.from_pretrained("Ganth1811/sino-nom-to-vietnamese-mbart-6epoch")
if forced_bos_token_id is not None:
    model.config.forced_bos_token_id = forced_bos_token_id

model.eval()
device = next(model.parameters()).device
@torch.no_grad()
def generate_predictions(input_texts, batch_size=16):
    predictions = []
    for i in tqdm(range(0, len(input_texts), batch_size), desc="Generating Predictions"):
        batch = [TASK_PREFIX + str(t) for t in input_texts[i:i + batch_size]]
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_SOURCE_LEN
        ).to(device)
        
        gen_kwargs = {"max_new_tokens": MAX_TARGET_LEN, "num_beams": 4}
        if forced_bos_token_id is not None:
            gen_kwargs["forced_bos_token_id"] = forced_bos_token_id
            
        outputs = model.generate(
            **inputs,
            max_new_tokens=64,             # Giới hạn độ dài output hợp lý
            num_beams=4,
            repetition_penalty=1.3,        # Phạt lặp từ (ngăn chặn hiện tượng lặp chuỗi)
            no_repeat_ngram_size=3,        # Không cho phép lặp cụm 3 từ
            early_stopping=True
        )
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        predictions.extend([d.strip() for d in decoded])

    return predictions

test_inputs = test_df[INPUT_COL].astype(str).tolist()
test_targets = test_df[TARGET_COL].astype(str).tolist()
test_preds = generate_predictions(test_inputs, batch_size=BATCH_SIZE)

# Metric Calculations
def calculate_exact_match(targets, preds):
    return np.mean([t.strip() == p.strip() for t, p in zip(targets, preds)])

def calculate_char_accuracy(targets, preds):
    correct, total = 0, 0
    for t, p in zip(targets, preds):
        min_len = min(len(t), len(p))
        correct += sum(1 for i in range(min_len) if t[i] == p[i])
        total += max(len(t), len(p))
    return correct / total if total > 0 else 0.0

def calculate_syllable_accuracy(targets, preds):
    correct, total = 0, 0
    for t, p in zip(targets, preds):
        t_words = t.strip().split()
        p_words = p.strip().split()
        min_len = min(len(t_words), len(p_words))
        correct += sum(1 for i in range(min_len) if t_words[i] == p_words[i])
        total += max(len(t_words), len(p_words))
    return correct / total if total > 0 else 0.0

em_score = calculate_exact_match(test_targets, test_preds)
ca_score = calculate_char_accuracy(test_targets, test_preds)
syl_acc = calculate_syllable_accuracy(test_targets, test_preds)

bleu_metric = evaluate.load("sacrebleu")
bleu_score = bleu_metric.compute(predictions=test_preds, references=[[t] for t in test_targets])["score"]

rouge_metric = evaluate.load("rouge")
rouge_score = rouge_metric.compute(predictions=test_preds, references=test_targets)["rougeL"]

print("\n" + "="*40)
print("          TEST EVALUATION RESULTS        ")
print("="*40)
print(f"Exact Match (EM):        {em_score * 100:.2f}%")
print(f"Character Accuracy (CA): {ca_score * 100:.2f}%")
print(f"Syllable Accuracy:      {syl_acc * 100:.2f}%")
print(f"BLEU Score:             {bleu_score:.2f}")
print(f"ROUGE-L Score:          {rouge_score * 100:.2f}%")
print("="*40)

# Save predictions
results_df = test_df.copy()
results_df["prediction"] = test_preds
PRED_CSV_PATH = OUTPUT_DIR / "test_predictions.csv"
results_df.to_csv(PRED_CSV_PATH, index=False)
print(f"Predictions saved to: {PRED_CSV_PATH.resolve()}")
display(results_df.head(10))

In [ ]:
import shutil

zip_path = "/kaggle/working/output/final_model"
shutil.make_archive(zip_path, 'zip', FINAL_MODEL_DIR)
print(f"Successfully created zip archive at: {zip_path}.zip")